## 🇧🇷 Brazilian Grand Prix | 2025

In this notebook we focus on debriefing the Brazilian GP, 2025. I am currently ironing out my high-level API and ingridients that compose the recipe of a race debrief and will continue to evolve over the next few notebooks.

### Debrief Recipe (4 Pillars)
- Car Setup and Circuit Characteristics corelation.
- Car Dynamics and Management Delta.
- Tyre Degradation Coefficients.
- Attention Flags for best Car and Driving Style (wrt Telemetry; if available).

### Notebook Configurations

In [1]:
from pathlib import Path
import sys


# Accessing the Current Working Directory for Absolute Root Path
REPO_ROOT_PATH = Path.cwd().parent.parent.parent

# Pathlib Path Objects for the Notebook
F1_PATH = REPO_ROOT_PATH / "F1"
CACHE_PATH = F1_PATH / "cache"

# Adding the source scripts into the Python Execution Path
if F1_PATH not in sys.path:
    sys.path.append(str(F1_PATH))

### Imports

In [2]:
# FastF1 Dependencies
import fastf1
from fastf1.core import Laps

# Visualisation Dependencies
from plotly.offline import init_notebook_mode

# Codebase Dependencies
from src.custom import CustomSession
from src.data import DataUtils
from src.pipeline import Pipeline
from src.plotting import Plotting
from src.utils import TELEMETRY_KEYPOINTS_BY_DIST

# Enabling Notebook Rendering for Plotly
init_notebook_mode(connected=True)

### Accessing the Specific Event and Sessions

In [3]:
# Defaulting the Requests for Caching to a Predefined path
# fastf1.Cache.clear_cache(cache_dir=str(CACHE_PATH))
fastf1.Cache.enable_cache(cache_dir=str(CACHE_PATH))

# Enabling Offline Mode
fastf1.Cache.offline_mode(enabled=True)

# The GP Event
brazil_2025 = fastf1.get_event(
    year=2025,
    gp="brazil",
    backend="fastf1"
)
print(brazil_2025)

RoundNumber                                                         21
Country                                                         Brazil
Location                                                     São Paulo
OfficialEventName    FORMULA 1 MSC CRUISES GRANDE PRÊMIO DE SÃO PAU...
EventDate                                          2025-11-09 00:00:00
EventName                                         São Paulo Grand Prix
EventFormat                                          sprint_qualifying
Session1                                                    Practice 1
Session1Date                                 2025-11-07 11:30:00-03:00
Session1DateUtc                                    2025-11-07 14:30:00
Session2                                             Sprint Qualifying
Session2Date                                 2025-11-07 15:30:00-03:00
Session2DateUtc                                    2025-11-07 18:30:00
Session3                                                        Sprint
Sessio

### Source Dependency Instances

In [4]:
# Data Handler: Loading and Storing Data from FastF1
data_handle = DataUtils(race_event=brazil_2025, cache_dir=str(CACHE_PATH))

# Data Pipeline: Transforming Data
data_pipeline = Pipeline()

# Plot Handler: Plotting Categorical, Joint, Marginal Distributions and Telemetry Traces
plot_handle = Plotting()

### Loading the Data for the Event

In [5]:
brazil_quali, brazil_race = data_handle.load_data()
print(brazil_quali)

core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 5
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 5)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '12', '16', '81', '6', '63', '30', '87', '10', '27', '14', '23', '44', 

2025 Season Round 21: São Paulo Grand Prix - Qualifying


### CustomSession objects for each FastF1 Session

In [6]:
# Accessing the different data streams for each session
qualifying_session = CustomSession(session=brazil_quali)
race_session = CustomSession(session=brazil_race)

### Data Cleaning and Segregation

**Accessing the top 5 drivers for each of the sessions**

In [7]:
top_5_drivers_in_race = race_session.results.iloc[:5]["DriverNumber"]
print("Top 5 Drivers in the Race:")
print(top_5_drivers_in_race)

top_5_drivers_in_quali = qualifying_session.results.iloc[:5]["DriverNumber"]
print("\nTop 5 Drivers in the Qualifying:")
print(top_5_drivers_in_quali)

Top 5 Drivers in the Race:
4      4
12    12
1      1
63    63
81    81
Name: DriverNumber, dtype: object

Top 5 Drivers in the Qualifying:
4      4
12    12
16    16
81    81
6      6
Name: DriverNumber, dtype: object


**Picking only the quick laps from the sessions for the top drivers (in the race)**<br><br>

⚠️ *As much as Will Buxton and DTS have gotten to me the reasoning is straight-forward points arent awarded for qualifying and hence all the analysis is done based on the results of the race*

✅ `However a separate Quali Lap Telemetry based dominance will also be provided in the analysis`

In [8]:
race_fast_laps = (
    race_session.laps
    .pick_drivers(identifiers=top_5_drivers_in_race)  
    .pick_quicklaps()
)

quali_fast_laps_by_quali = (
    qualifying_session.laps
    .pick_drivers(identifiers=top_5_drivers_in_quali)  
    .pick_quicklaps()
)

quali_fast_laps_by_race = (
    qualifying_session.laps
    .pick_drivers(identifiers=top_5_drivers_in_race)
    .pick_quicklaps()
)
quali_fast_laps_by_race.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
1,0 days 00:23:46.304000,NOR,4,0 days 00:01:10.404000,2.0,1.0,NaT,NaT,0 days 00:00:18.229000,0 days 00:00:36.070000,...,True,McLaren,0 days 00:22:35.900000,2025-11-08 18:08:27.336,1,NaN,False,,False,True
4,0 days 00:28:44.927000,NOR,4,0 days 00:01:10.249000,5.0,1.0,NaT,NaT,0 days 00:00:18.149000,0 days 00:00:36.079000,...,True,McLaren,0 days 00:27:34.678000,2025-11-08 18:13:26.114,1,NaN,False,,False,True
7,0 days 00:35:17.456000,NOR,4,0 days 00:01:09.656000,8.0,2.0,NaT,NaT,0 days 00:00:18.027000,0 days 00:00:35.681000,...,True,McLaren,0 days 00:34:07.800000,2025-11-08 18:19:59.236,1,NaN,False,,False,True
10,0 days 00:51:08.787000,NOR,4,0 days 00:01:09.930000,11.0,3.0,NaT,NaT,0 days 00:00:18.131000,0 days 00:00:35.804000,...,False,McLaren,0 days 00:49:58.857000,2025-11-08 18:35:50.293,1,NaN,False,,False,True
13,0 days 00:58:20.775000,NOR,4,0 days 00:01:09.616000,14.0,4.0,NaT,NaT,0 days 00:00:17.998000,0 days 00:00:35.587000,...,True,McLaren,0 days 00:57:11.159000,2025-11-08 18:43:02.595,1,NaN,False,,False,True


**Segregating the Telemetry among Mini-Sessions**<br><br>
⚠️ *Telemetry only available for Qualifying*

In [ ]:
# Splitting the quick laps among the mini qualifying sessions
q1, q2, q3 = quali_fast_laps_by_race.split_qualifying_sessions()

assert isinstance(q1, Laps)
assert isinstance(q2, Laps)
assert isinstance(q3, Laps)

# Lookup tables for the quick-lap driven telemetry
q1_telemetry_digest, q2_telemetry_digest, q3_telemetry_digest = {}, {}, {}

# Iterating through the top 5 drivers to access telemetetry from Qualifying
for driver_number in top_5_drivers_in_race:
    q1_telemetry = (
        q1.pick_drivers(identifiers=driver_number)
        .pick_fastest()
        .get_car_data()  # type: ignore
        .add_differential_distance()  # Delta Distance covered between two samples
        .add_distance()  # Cumulative Distance covered from first sample
    )
    q1_telemetry_digest[driver_number] = q1_telemetry

    q2_telemetry = (
        q2.pick_drivers(identifiers=driver_number)
        .pick_fastest()
        .get_car_data()
        .add_differential_distance()
        .add_distance()
    )
    q2_telemetry_digest[driver_number] = q2_telemetry

    q3_telemetry = (
        q3.pick_drivers(identifiers=driver_number)
        .pick_fastest()  
        .get_car_data()  # type: ignore
        .add_differential_distance()
        .add_distance()
    )
    q3_telemetry_digest[driver_number] = q3_telemetry

AttributeError: 'NoneType' object has no attribute 'get_car_data'

**Windowing Minisectors on the Track by Distance**

In [10]:
# Iterating through the driver telemetry for each session to apply windowing
for driver_number in top_5_drivers_in_quali:
    
    # Making a copy to avoid pd.futurewarning
    # Mapping the telemetry keypoints for Q1
    copy_tel_q1 = q1_telemetry_digest[driver_number].copy()
    copy_tel_q1 = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q1)
    q1_telemetry_digest[driver_number] = copy_tel_q1

    # Mapping the telemetry keypoints for Q2
    copy_tel_q2 = q2_telemetry_digest[driver_number].copy()
    copy_tel_q2 = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q2)
    q2_telemetry_digest[driver_number] = copy_tel_q2

    # Mapping the telemetry keypoints for Q3
    copy_tel_q3 = q3_telemetry_digest[driver_number].copy()
    copy_tel_q3 = data_pipeline.map_telemetry_keypoints(copy_frame=copy_tel_q3)
    q3_telemetry_digest[driver_number] = copy_tel_q3

# Testing the output transformation
q3_telemetry_digest["81"].head()

,Date,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Time,SessionTime,DifferentialDistance,Distance,Keypoint
0,2025-11-08 19:03:49.980,11739.0,334.0,8,100.0,False,12,car,0 days 00:00:00.027000,0 days 01:17:58.544000,2.505000,2.505000,Lap_start_top_speed
1,2025-11-08 19:03:50.340,11749.0,334.0,8,100.0,False,12,car,0 days 00:00:00.387000,0 days 01:17:58.904000,33.400000,35.905000,Lap_start_top_speed
2,2025-11-08 19:03:50.580,11784.0,335.0,8,100.0,False,12,car,0 days 00:00:00.627000,0 days 01:17:59.144000,22.333333,58.238333,Lap_start_top_speed
3,2025-11-08 19:03:50.820,11803.0,335.0,8,100.0,False,12,car,0 days 00:00:00.867000,0 days 01:17:59.384000,22.333333,80.571667,Lap_start_top_speed
4,2025-11-08 19:03:51.059,11815.0,336.0,8,100.0,False,12,car,0 days 00:00:01.106000,0 days 01:17:59.623000,22.306667,102.878333,T1_braking_stability


**Visualising a sample telemetry trace**

In [11]:
plot_handle.plot_driver_telemetry_traces(
    top_5_driver_telemetry=q3_telemetry_digest
)

### Feature Engineering

**Qualifying Lap Throttle Percentage**<br><br>
⚠️ *Based on race results to calculate throttle usage per lap*

In [12]:
throttle_map_digest = {}

# Accessing the full Q1 laps for throttle mapping drivers based on race results
q1_full_laps, _, _ = qualifying_session.laps.split_qualifying_sessions()
assert isinstance(q1_full_laps, Laps)

# Iterating through the drivers to calculate the driver throttle maps
for driver_number in top_5_drivers_in_race:
    throttle_map = data_handle.get_throttle_map(
        driver_number=driver_number,
        driver_quali_laps=q1_full_laps
    )
    throttle_map_digest[driver_number] = throttle_map

throttle_map_digest

{'4': (0.5648854961832062, 0.3969465648854961, 0.03816793893129771),
 '12': (0.5836431226765799, 0.379182156133829, 0.03717472118959108),
 '1': (0.5904059040590406, 0.3726937269372694, 0.03690036900369004),
 '63': (0.5805243445692884, 0.3820224719101124, 0.03745318352059925),
 '81': (0.5622641509433962, 0.4, 0.03773584905660377)}

**Fuel Aware Laptime Correction**

In [13]:
# Initialising the new columns
race_fast_laps["LapFuelBurn"] = 0.0
race_fast_laps["CumulativeFuelBurn"] = 0.0
race_fast_laps["FuelAwareLapTime"] = 0.0

# Iterating through the drivers to correct the laptimes
for driver_number in top_5_drivers_in_race:
    
    # Accessing the Drivers Laps and Throttle Map
    driver_laps = race_fast_laps[race_fast_laps["DriverNumber"] == driver_number].copy()
    driver_throttle_map = throttle_map_digest[driver_number]

    # Tranforming the frame for the Fuel Aware Laptimes
    corrected_laptimes = data_pipeline.get_fuel_aware_laptime(
        driver_laps=driver_laps,
        driver_throttle_handle=driver_throttle_map
    )

    # Moving the changes to the orignal laps frame
    race_fast_laps[race_fast_laps["DriverNumber"] == driver_number] = corrected_laptimes

race_fast_laps.groupby("Driver")["CumulativeFuelBurn"].last()

Driver
ANT    88.979824
NOR    87.614126
PIA    87.568726
RUS    87.396732
VER    89.262118
Name: CumulativeFuelBurn, dtype: float64

**Efficiency Factor: Top Speed Vs Avg Speed (Cornering Load)**

In [ ]:
assert isinstance(q1_full_laps, Laps)

for driver_number in top_5_drivers_in_race:
    driver_laps = (
        q1_full_laps.pick_drivers(identifiers=driver_number)
        .pick_fastest()
        .get_car_data()  # type: ignore
        .add_distance()
    )
    pia_laps = data_pipeline.map_telemetry_keypoints(pia_laps)

,Date,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Time,SessionTime,Distance,Keypoint
0,2025-11-08 18:13:49.929,11733.0,333.0,8,100.0,False,12,car,0 days 00:00:00.178000,0 days 00:27:58.493000,16.465000,Lap_start_top_speed
1,2025-11-08 18:13:50.129,11739.0,334.0,8,100.0,False,12,car,0 days 00:00:00.378000,0 days 00:27:58.693000,35.020556,Lap_start_top_speed
2,2025-11-08 18:13:50.409,11753.0,334.0,8,100.0,False,12,car,0 days 00:00:00.658000,0 days 00:27:58.973000,60.998333,Lap_start_top_speed
3,2025-11-08 18:13:50.809,11777.0,336.0,8,100.0,False,12,car,0 days 00:00:01.058000,0 days 00:27:59.373000,98.331667,Lap_start_top_speed
4,2025-11-08 18:13:51.089,11815.0,336.0,8,100.0,False,12,car,0 days 00:00:01.338000,0 days 00:27:59.653000,124.465000,T1_braking_stability
...,...,...,...,...,...,...,...,...,...,...,...,...
260,2025-11-08 18:14:58.609,11617.0,330.0,8,100.0,False,12,car,0 days 00:01:08.858000,0 days 00:29:07.173000,4143.915278,Home_top_speed
261,2025-11-08 18:14:58.810,11610.0,330.0,8,100.0,False,12,car,0 days 00:01:09.059000,0 days 00:29:07.374000,4162.340278,Home_top_speed
262,2025-11-08 18:14:59.130,11610.0,330.0,8,100.0,False,12,car,0 days 00:01:09.379000,0 days 00:29:07.694000,4191.673611,Home_top_speed
263,2025-11-08 18:14:59.290,11610.0,330.0,8,100.0,False,12,car,0 days 00:01:09.539000,0 days 00:29:07.854000,4206.340278,Home_top_speed


In [43]:
index_corr = pia_laps.groupby("Keypoint", observed=False)["Speed"].agg(["mean", "max"])
index_corr

,mean,max
Keypoint,,
Lap_start_top_speed,334.250000,336.0
T1_braking_stability,334.000000,338.0
T1_in,145.629630,255.0
T2_out,254.823529,294.0
T3_acc,315.842105,328.0
T4_in,238.285714,310.0
T5_out,215.777778,261.0
T5_acc,281.307692,292.0
T6_in,248.250000,288.0


In [ ]:
avg_load_keys = [
    "T1_in", "T2_out", "T4_in", "T5_out", "T6_in", 
    "T7_out", "T8_in", "T8_out", "T9_in", "T9_out",
    "T10_in", "T10_out", "T12_in", "T12_out"
]
top_speed_keys = [
    'Lap_start_top_speed', 'T1_braking_stability', 'T3_acc', 'T5_acc', 
    'T10_acc', 'T11_load', 'T13_load', 'Home_acc', 'Home_top_speed'
]


avg_load_keys_mean = sum([index_corr.loc[key]["mean"] for key in avg_load_keys]) / len(avg_load_keys)
avg_load_keys_max = sum([index_corr.loc[key]["max"] for key in avg_load_keys]) / len(avg_load_keys)
top_speed_keys_mean = sum([index_corr.loc[key]["mean"] for key in top_speed_keys]) / len(top_speed_keys)
top_speed_keys_max = sum([index_corr.loc[key]["max"] for key in top_speed_keys]) / len(top_speed_keys)

print(f"Average Efficiency: {avg_load_keys_mean/top_speed_keys_mean}")
print(f"Max Efficiency: {avg_load_keys_max/top_speed_keys_max}")

Average Efficiency: 0.6103508054871136
Max Efficiency: 0.7424314096499527


## Fin 🏎️ ✨